# Data gaps

Measures how complete the compendium actually is, and feeds the dashboard in
`docs/`.

Reads the final English long files, `COMPENDIUM-ARAB SOCIETY\<Chapter>_EN.xlsx`,
and writes:

| Output | What it is |
| --- | --- |
| `data_gaps_report.xlsx` | every metric and every flagged figure, one sheet each |
| `docs/data.js` | the dashboard's data, regenerated |

## Why not just a fill rate

Only 9&ndash;18% of rows in these files carry a value, but that number is
meaningless: the questionnaire grid is every age group &times; marital status
&times; nationality &times; year whether or not the combination was ever
intended to be filled. Reporting it as "14% complete" would be wrong.

So completeness is measured at two levels instead:

**Level 1 &mdash; how many points each series has.** For each indicator &times;
country, how many of the 17 expected years (2010&ndash;2026) carry a value, and
separately how many carry a *headline* value with every breakdown at its total.

Reported as a **distribution**, not an average: how many series have all 17
points, how many have 16, 15, and so on down to 0. One mean percentage would let
a one-point series and a full one cancel out into a number that reads as "half
full", which is exactly the false impression to avoid.

Every indicator &times; country pair is counted, including the ones that never
reported anything &mdash; 297 of 1,112 on the current data, 27%. Leaving those
out of the denominator flatters every summary built on top.

**Level 2 &mdash; disaggregation availability.** Which breakdowns a country
actually supplies for an indicator. This answers "does Oman report by
nationality?".

Seven dimensions &mdash; causes of death, economic activity, employment status,
main occupation, institutional sector, reasons for inactivity, the ICD list
&mdash; have **no total category**, because an indicator broken down that way has
no meaningful aggregate row. Headline coverage is therefore blank for those, by
design rather than as a gap.


## Config


In [ ]:
"""
CELL: Configuration.
"""
import json
import re
from pathlib import Path

import pandas as pd

COMPENDIUM_PATH = Path(r"C:\Users\RSHIRINI\OneDrive - United Nations\Desktop\DSS\COMPENDIUM-ARAB SOCIETY")
CODES_PATH = COMPENDIUM_PATH / "codes"
PROJECT_PATH = CODES_PATH / "data quality"
# The dashboard sits at the repository root, not beside this notebook:
# GitHub Pages can only serve a branch root or a /docs at that root.
DOCS_PATH = CODES_PATH / "docs"
REPORT_PATH = PROJECT_PATH / "data_gaps_report.xlsx"

# The years the compendium expects a figure for. 2027-2030 exist as projection
# columns in the template and are not counted as gaps.
FIRST_YEAR, LAST_YEAR = 2010, 2026
EXPECTED_YEARS = list(range(FIRST_YEAR, LAST_YEAR + 1))

# Columns that are not a disaggregation.
FIXED = {"Indicator", "Country", "Chapter", "Year", "Value", "Source"}

# The value meaning "all of them", per dimension. A dimension absent from this
# map has no total by design - see the note above.
TOTAL_LABELS = {
    "Sex": "Both sexes",
    "Age Group": "Age Total",
    "Area": "Area Total",
    "Nationality": "Nationality Total",
    "Marital status": "Marital status Total",
    "Quintile": "Total",
    "Types of products/services": "Total",
}

# A year-on-year change of more than this is flagged. Deliberately loose: real
# populations do not change tenfold in a year, so a hit is a transcription
# error, a units change, or a break in the series worth a footnote.
SPIKE_FACTOR = 10

# How far a reported total may stray from the sum of its parts, as a fraction.
TOTAL_TOLERANCE = 0.01

# The dashboard carries the worst N; the Excel report carries all of them.
MAX_DASHBOARD_ISSUES = 2000

# Series are grouped by how many of the 17 expected years they actually carry.
# Reporting one average percentage would let a one-point series and a full one
# cancel out into a number that reads as "half full"; these bands keep the
# difference visible. Ordered worst to best.
POINT_BANDS = [
    ("none", 0, 0),          # never reported at all
    ("sparse", 1, 6),        # too few points to read as a series
    ("partial", 7, 12),
    ("strong", 13, 16),
    ("complete", 17, 17),    # every expected year present
]


def band_of(points):
    for name, low, high in POINT_BANDS:
        if low <= points <= high:
            return name
    return "none"


## Helpers


In [ ]:
"""
CELL: Helpers.
"""


def to_number(value):
    """Parse one Value cell into a float, or None if there is no number in it.

    These files store figures as text more often than as numbers: ' 701 956 '
    uses spaces as thousand separators, some cells use commas, some carry a
    non-breaking space, and a few hold '-' for "no data".
    """
    if pd.isna(value):
        return None
    text = str(value).replace("\xa0", " ").replace(",", "").strip()
    text = re.sub(r"\s+", "", text)
    if text in ("", "-", "--", "..", "..."):
        return None
    try:
        return float(text)
    except ValueError:
        return None


def headline_mask(table, dimensions):
    """True where a row is the headline figure: every dimension either empty or
    sitting at its own total. A dimension with no total label must be empty."""
    mask = pd.Series(True, index=table.index)
    for dimension in dimensions:
        column = table[dimension]
        total = TOTAL_LABELS.get(dimension)
        ok = column.isna()
        if total is not None:
            ok = ok | (column.astype(str).str.strip() == total)
        mask &= ok
    return mask


def missing_years_and_longest_gap(present):
    """Which expected years are absent, and the longest unbroken run of them."""
    missing = [y for y in EXPECTED_YEARS if y not in present]
    longest = run = 0
    for year in EXPECTED_YEARS:
        run = run + 1 if year not in present else 0
        longest = max(longest, run)
    return missing, longest


## Measuring one chapter

Level 1 and level 2 completeness, plus the three kinds of flagged figure:
dubious year-on-year spikes, percentage indicators holding counts, and
totals that contradict the sum of their own parts.


In [ ]:
"""
CELL: measure_chapter() - every metric for one chapter's long file.
"""


def measure_chapter(path):
    """Returns (series, dimensions, issues) for one <Chapter>_EN.xlsx."""
    chapter = path.name[: -len("_EN.xlsx")]
    table = pd.read_excel(path, engine="openpyxl")
    if "Indicator" not in table.columns:
        logger.warning(f"{path.name}: no Indicator column, skipping")
        return [], [], []

    dimensions = [c for c in table.columns if c not in FIXED]
    table["number"] = table["Value"].map(to_number)
    have = table[table["number"].notna()].copy()
    have["Year"] = have["Year"].astype(int)
    have["is_headline"] = headline_mask(have, dimensions)

    logger.info(f"{chapter}: {len(table):,} rows, {len(have):,} with a value, "
                f"{len(dimensions)} dimension(s)")

    series, dimension_rows, issues = [], [], []

    # -------------------------------------------------- levels 1 and 2
    # Every indicator x country the chapter could report, not only the pairs
    # that reported something. A pair with nothing is a real finding - a
    # 0-point series - and leaving it out of the denominator would quietly
    # flatter every summary built on top.
    reported = {key: group for key, group
                in have.groupby(["Indicator", "Country"], observed=True)}
    all_indicators = sorted(table["Indicator"].dropna().unique())
    all_countries = sorted(table["Country"].dropna().unique())

    for indicator in all_indicators:
        for country in all_countries:
            group = reported.get((indicator, country))

            if group is None:
                years_any, years_head = set(), set()
            else:
                years_any = set(group["Year"]) & set(EXPECTED_YEARS)
                years_head = set(group.loc[group["is_headline"], "Year"]) & set(EXPECTED_YEARS)
            missing, longest = missing_years_and_longest_gap(years_any)

            series.append({
                "chapter": chapter, "indicator": indicator, "country": country,
                "points": len(years_any),
                "headline_points": len(years_head),
                "band": band_of(len(years_any)),
                "coverage": round(len(years_any) / len(EXPECTED_YEARS) * 100, 1),
                "first_year": min(years_any) if years_any else None,
                "last_year": max(years_any) if years_any else None,
                "longest_gap": longest, "missing_years": missing,
                "values": 0 if group is None else len(group),
            })

            if group is None:
                continue
            for dimension in dimensions:
                total = TOTAL_LABELS.get(dimension)
                values = group[dimension].dropna().astype(str).str.strip()
                if total is not None:
                    values = values[values != total]
                if values.empty:
                    continue
                dimension_rows.append({
                    "chapter": chapter, "indicator": indicator, "country": country,
                    "dimension": dimension, "categories": values.nunique(),
                })

    # ------------------------------------------------------- dubious spikes
    # One value per series per year first: the same dimensions and year can
    # appear twice in these files, and without this the comparison below would
    # pit two rows from the SAME year against each other.
    keys = ["Indicator", "Country"] + dimensions
    tidy = (have.groupby(keys + ["Year"], dropna=False, observed=True)["number"]
            .first().reset_index().sort_values(keys + ["Year"]))
    grouped = tidy.groupby(keys, dropna=False, observed=True)
    tidy["previous"] = grouped["number"].shift(1)
    tidy["previous_year"] = grouped["Year"].shift(1)
    candidates = tidy[tidy["previous"].notna() & (tidy["previous"] != 0)
                      & (tidy["Year"] > tidy["previous_year"])].copy()
    ratio = candidates["number"] / candidates["previous"]
    for _, row in candidates[(ratio > SPIKE_FACTOR) | (ratio < 1 / SPIKE_FACTOR)].iterrows():
        issues.append({
            "chapter": chapter, "kind": "spike",
            "indicator": row["Indicator"], "country": row["Country"],
            "detail": (f"{int(row['previous_year'])}: {row['previous']:,.0f}"
                       f"  ->  {int(row['Year'])}: {row['number']:,.0f}"),
            "severity": round(abs(row["number"] / row["previous"]), 1),
        })

    # ------------------------------- percentage indicators holding counts
    name = have["Indicator"].astype(str).str.lower()
    percentages = have[name.str.contains(r"%|percentage|proportion", regex=True, na=False)]
    for indicator, group in percentages.groupby("Indicator", observed=True):
        over = group["number"] > 100
        if over.any() and not over.all():
            issues.append({
                "chapter": chapter, "kind": "units",
                "indicator": indicator, "country": "(several)",
                "detail": (f"{int(over.sum()):,} of {len(group):,} rows exceed 100 "
                           f"(largest {group['number'].max():,.0f}) - percentages and "
                           f"counts mixed in one indicator"),
                "severity": round(float(group["number"].max()), 0),
            })

    # ------------------------------ totals that contradict their own parts
    for dimension, total in TOTAL_LABELS.items():
        if dimension not in have.columns:
            continue
        keys2 = [c for c in ["Indicator", "Country", "Year", "Sex"]
                 if c in have.columns and c != dimension]
        work = have[["number", dimension] + keys2]
        label = work[dimension].astype(str).str.strip()
        totals = work[label == total].groupby(keys2, observed=True)["number"].first()
        parts = (work[work[dimension].notna() & (label != total) & (label != "Age unknown")]
                 .groupby(keys2, observed=True)["number"].sum(min_count=1))
        shared = totals.index.intersection(parts.index)
        if not len(shared):
            continue
        compare = pd.DataFrame({"total": totals.loc[shared], "parts": parts.loc[shared]})
        compare = compare[compare["total"].abs() > 0]
        gap = (compare["parts"] - compare["total"]).abs() / compare["total"].abs()
        for key, off in gap[gap > TOTAL_TOLERANCE].items():
            record = dict(zip(keys2, key if isinstance(key, tuple) else (key,)))
            issues.append({
                "chapter": chapter, "kind": "total mismatch",
                "indicator": record.get("Indicator", ""),
                "country": record.get("Country", ""),
                "detail": (f"{dimension} {record.get('Year', '')}: reported "
                           f"{compare.loc[key, 'total']:,.0f} vs parts summing to "
                           f"{compare.loc[key, 'parts']:,.0f}"),
                "severity": round(float(off) * 100, 1),
            })

    return series, dimension_rows, issues


## Run


In [ ]:
"""
CELL: Measure every chapter.
"""
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s",
                    datefmt="%H:%M:%S")
logger = logging.getLogger("data-gaps")

files = sorted(COMPENDIUM_PATH.glob("*_EN.xlsx"))
print(f"Reading {len(files)} chapter file(s) from {COMPENDIUM_PATH}\n")

all_series, all_dimensions, all_issues = [], [], []
for path in files:
    s, d, i = measure_chapter(path)
    all_series += s
    all_dimensions += d
    all_issues += i

SERIES = pd.DataFrame(all_series)
DIMENSIONS = pd.DataFrame(all_dimensions)
ISSUES = pd.DataFrame(all_issues)

if SERIES.empty:
    print("\nNo data. Run the pipeline notebooks first.")
else:
    print(f"\n{len(SERIES):,} series · {len(DIMENSIONS):,} disaggregation entries "
          f"· {len(ISSUES):,} flagged figures")

    print(f"\nHow many of the {len(EXPECTED_YEARS)} years each series actually has:")
    counts = SERIES["points"].value_counts().sort_index(ascending=False)
    for points, n in counts.items():
        bar = "#" * max(1, round(n / counts.max() * 40))
        note = "  <- complete" if points == len(EXPECTED_YEARS) else (
               "  <- nothing at all" if points == 0 else "")
        print(f"   {points:>2} points  {n:>5,}  {bar}{note}")

    print("\nBy band:")
    for name, low, high in POINT_BANDS:
        n = (SERIES["band"] == name).sum()
        span = f"{low}" if low == high else f"{low}-{high}"
        print(f"   {name:<9} ({span:>5} points)  {n:>5,}  {n/len(SERIES):>5.1%}")

    print("\nComplete series by chapter:")
    for chapter, group in SERIES.groupby("chapter"):
        full = (group["points"] == len(EXPECTED_YEARS)).sum()
        none = (group["points"] == 0).sum()
        print(f"   {chapter:<12} {full:>4,} complete · {none:>4,} empty · "
              f"{len(group):>4,} total")

    print(f"\nSeries reporting nothing since 2019: "
          f"{(SERIES['last_year'] < 2020).sum():,}")
    if len(ISSUES):
        print("\nFlagged figures by kind:")
        print(ISSUES["kind"].value_counts().to_string())


## The Excel report


In [ ]:
"""
CELL: Write the Excel report.
"""
if not SERIES.empty:
    with pd.ExcelWriter(REPORT_PATH, engine="openpyxl") as writer:
        summary = pd.DataFrame(
            [{"metric": "series (every indicator x country)", "value": len(SERIES)}]
            + [{"metric": f"{name} ({low} points)" if low == high
                          else f"{name} ({low}-{high} points)",
                "value": int((SERIES["band"] == name).sum())}
               for name, low, high in reversed(POINT_BANDS)]
            + [{"metric": "series stale since 2019",
                "value": int((SERIES["last_year"] < 2020).sum())},
               {"metric": "flagged figures", "value": len(ISSUES)}])
        summary.to_excel(writer, sheet_name="summary", index=False)

        # One row per point count, so the shape of the distribution is readable
        # without opening the dashboard.
        distribution = (SERIES["points"].value_counts().reindex(
            range(len(EXPECTED_YEARS), -1, -1), fill_value=0)
            .rename_axis("points").reset_index(name="series"))
        distribution["share"] = (distribution["series"] / len(SERIES)).round(3)
        distribution.to_excel(writer, sheet_name="points distribution", index=False)

        (SERIES.pivot_table(index="country", columns="band", values="indicator",
                            aggfunc="count", fill_value=0)
         .reindex(columns=[b[0] for b in reversed(POINT_BANDS)], fill_value=0)
         .to_excel(writer, sheet_name="bands by country"))
        (SERIES.pivot_table(index="indicator", columns="band", values="country",
                            aggfunc="count", fill_value=0)
         .reindex(columns=[b[0] for b in reversed(POINT_BANDS)], fill_value=0)
         .to_excel(writer, sheet_name="bands by indicator"))

        out = SERIES.copy()
        out["missing_years"] = out["missing_years"].map(
            lambda years: ", ".join(str(y) for y in years))
        out.sort_values("points").to_excel(writer, sheet_name="series", index=False)


        if not DIMENSIONS.empty:
            DIMENSIONS.to_excel(writer, sheet_name="disaggregation", index=False)
        if not ISSUES.empty:
            (ISSUES.sort_values("severity", ascending=False)
             .to_excel(writer, sheet_name="flagged figures", index=False))

    print(f"Wrote {REPORT_PATH.name}")
    print("  sheets: summary, points distribution, series, bands by country,")
    print("          bands by indicator, disaggregation, flagged figures")


## The dashboard

Regenerates `docs/data.js`. `docs/index.html` is static and does not need
rebuilding &mdash; open it directly, or publish the `docs/` folder with GitHub
Pages.


In [ ]:
"""
CELL: Regenerate the dashboard's data file.
"""


def write_dashboard_data():
    """Write docs/data.js as index-referenced arrays.

    Repeating "chapter"/"indicator"/"country" as JSON keys across 815 series and
    nearly 8,000 issues costs about 1.5 MB; the same content as arrays with a
    lookup table is a fraction of that, and the page rehydrates it on load.
    """
    def index_of(values):
        ordered = sorted(set(values))
        return ordered, {v: i for i, v in enumerate(ordered)}

    chapters, chapter_ix = index_of(SERIES["chapter"])
    countries, country_ix = index_of(SERIES["country"])
    indicators, indicator_ix = index_of(SERIES["indicator"])
    dimensions, dimension_ix = (index_of(DIMENSIONS["dimension"])
                                if not DIMENSIONS.empty else ([], {}))
    kinds, kind_ix = index_of(ISSUES["kind"]) if not ISSUES.empty else ([], {})

    series_rows = [[
        chapter_ix[r.chapter], indicator_ix[r.indicator], country_ix[r.country],
        int(r.points), int(r.headline_points),
        int(r.first_year) if pd.notna(r.first_year) else None,
        int(r.last_year) if pd.notna(r.last_year) else None,
        int(r.longest_gap), int(r.values), [int(y) for y in (r.missing_years or [])],
    ] for r in SERIES.itertuples(index=False)]

    dimension_rows = [[
        chapter_ix[r.chapter], indicator_ix[r.indicator], country_ix[r.country],
        dimension_ix[r.dimension], int(r.categories),
    ] for r in DIMENSIONS.itertuples(index=False)] if not DIMENSIONS.empty else []

    worst = (ISSUES.sort_values("severity", ascending=False).head(MAX_DASHBOARD_ISSUES)
             if not ISSUES.empty else ISSUES)
    issue_rows = [[
        chapter_ix.get(r.chapter, 0), kind_ix[r.kind],
        indicator_ix.get(r.indicator, -1), country_ix.get(r.country, -1),
        float(r.severity), str(r.detail),
    ] for r in worst.itertuples(index=False)] if not worst.empty else []

    payload = {
        "generated": pd.Timestamp.now().strftime("%d %B %Y"),
        "years": EXPECTED_YEARS,
        "chapters": chapters, "countries": countries,
        "indicators": indicators, "dimensions": dimensions, "kinds": kinds,
        "series": series_rows, "dims": dimension_rows, "issues": issue_rows,
        "issueCounts": ({k: int(v) for k, v in ISSUES["kind"].value_counts().items()}
                        if not ISSUES.empty else {}),
        "issuesShown": len(issue_rows), "issuesTotal": int(len(ISSUES)),
        "bands": [[name, low, high] for name, low, high in POINT_BANDS],
    }

    DOCS_PATH.mkdir(parents=True, exist_ok=True)
    path = DOCS_PATH / "data.js"
    path.write_text("window.GAPS = " + json.dumps(payload, separators=(",", ":")) + ";\n",
                    encoding="utf-8")
    return path


if not SERIES.empty:
    written = write_dashboard_data()
    print(f"Wrote {written.relative_to(PROJECT_PATH)} "
          f"({written.stat().st_size / 1024:,.0f} KB)")
    print("\nOpen docs/index.html to view the dashboard, or push docs/ to GitHub Pages.")
